In [0]:
%run ../../Include/DatabricksHelpers

In [0]:
%run ../../Include/silver_utils

In [0]:
# Cell 1: Setup, Context & Environment Initialization
db_helpers = DatabricksHelpers(dbutils)
current_user_email = db_helpers.current_user()

catalog_name = "f1_dev"
schemas = ["bronze", "silver", "gold"]

# Ensure bronze, silver, and gold schemas exist in Unity Catalog
initialize_environment(spark, catalog_name, schemas)

source_table = f"{catalog_name}.bronze.lap_times"
target_table = f"{catalog_name}.silver.fact_lap_times"

# --- CELL 1 TEST ---
assert current_user_email and ("@" in current_user_email or current_user_email == "root"), \
    f"TEST FAILED: Invalid user context derived: '{current_user_email}'"

assert spark.catalog.tableExists(source_table), \
    f"TEST FAILED: Source bronze table '{source_table}' does not exist in catalog."

print(f"✓ Cell 1 Passed: Executed by '{current_user_email}' on catalog '{catalog_name}'.")

In [0]:
# Cell 2: Data Extraction & Bronze String Sanitization
df_bronze = spark.table(source_table)

# Reusable string cleaning: purges '\N', 'NULL', 'null', and empty strings across all text fields
df_sanitized = sanitize_bronze_strings(df_bronze)

# --- CELL 2 TEST ---
string_cols = [f.name for f in df_sanitized.schema.fields if f.dataType.typeName() == "string"]
for col_name in string_cols:
    dirty_count = df_sanitized.filter(
        (F.col(col_name) == r"\N") | (F.lower(F.col(col_name)) == "null")
    ).count()
    assert dirty_count == 0, f"TEST FAILED: Found {dirty_count} unhandled string literals in '{col_name}'"

print(f"✓ Cell 2 Passed: String sanitization verified across {len(string_cols)} string columns.")

In [0]:
# Cell 3: Silver Domain Transformations & Safe Casting
df_lap_times_silver = (
    df_sanitized
    .select(
        F.col("raceId").try_cast("integer").alias("race_id"),
        F.col("driverId").try_cast("integer").alias("driver_id"),
        F.col("lap").try_cast("integer").alias("lap_number"),
        F.col("position").try_cast("integer").alias("running_position"),
        F.col("time").alias("lap_time_formatted"),
        F.col("milliseconds").try_cast("long").alias("lap_time_ms")
    )
    # Deduplicate against composite primary key: race_id + driver_id + lap_number
    .dropDuplicates(["race_id", "driver_id", "lap_number"])
)

# --- CELL 3 TEST ---
sample_row = df_lap_times_silver.first()

# 1. Foreign & Composite Key data type verification
assert dict(df_lap_times_silver.dtypes)["race_id"] == "int", "TEST FAILED: race_id cast failed."
assert dict(df_lap_times_silver.dtypes)["driver_id"] == "int", "TEST FAILED: driver_id cast failed."
assert dict(df_lap_times_silver.dtypes)["lap_number"] == "int", "TEST FAILED: lap_number cast failed."

# 2. Metric data type verification
assert dict(df_lap_times_silver.dtypes)["lap_time_ms"] == "bigint", "TEST FAILED: lap_time_ms cast failed."

print("✓ Cell 3 Passed: Lap times transformations and safe casts verified.")

In [0]:
# Cell 4: Pre-Write Data Quality Circuit Breaker & Domain Rules
composite_pk = ["race_id", "driver_id", "lap_number"]

# 1. Standard Circuit Breaker: Non-Null PK & Composite Key Uniqueness assertion
run_circuit_breaker(
    df=df_lap_times_silver, 
    primary_keys=composite_pk, 
    prohibited_pii=set()
)

# 2. Domain Rule Assertions: Lap times and positions must be strictly positive
invalid_times = df_lap_times_silver.filter(F.col("lap_time_ms") <= 0).count()
assert invalid_times == 0, f"CIRCUIT BREAKER: Found {invalid_times} records with lap_time_ms <= 0!"

invalid_positions = df_lap_times_silver.filter(F.col("running_position") <= 0).count()
assert invalid_positions == 0, f"CIRCUIT BREAKER: Found {invalid_positions} records with running_position <= 0!"

print("✓ Cell 4 Passed: Pre-write circuit breaker and lap time domain assertions passed.")

In [0]:
# Cell 5: Target Delta DDL Creation & Idempotent Delta Load
# 1. Ensure target Silver Fact Delta table exists with Composite Primary Key constraint
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        race_id INT NOT NULL,
        driver_id INT NOT NULL,
        lap_number INT NOT NULL,
        running_position INT,
        lap_time_formatted STRING,
        lap_time_ms BIGINT,
        CONSTRAINT fact_lap_times_pk PRIMARY KEY (race_id, driver_id, lap_number)
    )
    USING DELTA
""")

# 2. Reusable Upsert: Dynamically handles composite key MERGE logic and enables CDF
upsert_silver_delta(
    spark=spark, 
    df=df_lap_times_silver, 
    target_table=target_table, 
    primary_keys=composite_pk
)

# --- CELL 5 TEST ---
target_df = spark.table(target_table)
staging_count = df_lap_times_silver.count()
target_count = target_df.count()

# Confirm written count matches staging expectations
assert target_count >= staging_count, \
    f"TEST FAILED: Target table count ({target_count}) is lower than staging count ({staging_count})."

print(f"✓ Cell 5 Passed: Delta MERGE load to '{target_table}' completed and verified.")

In [0]:
# Cell 6: Unity Catalog Metadata Governance & Exit
# Reusable Governance Tagger: Sets tags and asserts existence in information_schema
apply_silver_tags(
    spark=spark, 
    active_catalog=catalog_name, 
    target_table=target_table, 
    current_user_email=current_user_email, 
    table_type="fact"
)

# Clean exit for Databricks Workflows / Jobs Orchestration
dbutils.notebook.exit(f"SUCCESS: Pipeline executed cleanly by {current_user_email} on {target_table}")